In [1]:
import numpy as np
import torch

In [4]:
import torchvision
from torchvision import transforms

transform = transforms.Compose([
             transforms.ToTensor(),
             transforms.Normalize((.5,), (.5,))
])
    
train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

trainLoader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
testLoaderr = torch.utils.data.DataLoader(test, batch_size=128, shuffle=False)




100%|██████████| 26.4M/26.4M [00:08<00:00, 3.05MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 140kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.45MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 16.9MB/s]


In [10]:
train.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [36]:
sample_img, sample_label = next(iter(trainLoader))
print(sample_label.shape)
sample_img.shape

torch.Size([128])


torch.Size([128, 1, 28, 28])

In [32]:
import torch.nn as nn
import torch.nn.functional as F
class MyCNN(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64*5*5, 512)
        self.fc2 = nn.Linear(512, 10)



    def forward(self, x):
        print(x.shape)

        x = self.pool(F.relu(self.conv1(x)))
        print(x.shape)

        x = self.pool(F.relu(self.conv2(x)))
        print(x.shape)

        x = torch.flatten(x, 1)
        print(x.shape)
        
        x = F.relu(self.fc1(x))
        print(x.shape)

        x = self.fc2(x)
        print(x.shape)
        return x

In [33]:
model = MyCNN()

print(model)

MyCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [34]:
dummy = torch.randn(128, 1, 28, 28)
model(dummy)

torch.Size([128, 1, 28, 28])
torch.Size([128, 32, 13, 13])
torch.Size([128, 64, 5, 5])
torch.Size([128, 1600])
torch.Size([128, 512])
torch.Size([128, 10])


tensor([[-0.0448,  0.0139, -0.1577,  ..., -0.1277, -0.0042,  0.1298],
        [-0.0513,  0.0226, -0.1420,  ..., -0.0955, -0.0029,  0.1911],
        [-0.0070,  0.0575, -0.1601,  ..., -0.0688,  0.0148,  0.1876],
        ...,
        [-0.0373,  0.0290, -0.1355,  ..., -0.0636,  0.0142,  0.1568],
        [-0.0204,  0.0960, -0.1734,  ..., -0.1496, -0.0206,  0.1428],
        [-0.0750,  0.0782, -0.1693,  ..., -0.1847, -0.0216,  0.2275]],
       grad_fn=<AddmmBackward0>)

In [ ]:
from torch import optim
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fun = nn.CrossEntropyLoss()

for epoch in range(2):
    correct = 0
    total = 0
    for img, tar in trainLoader:
        optimizer.zero_grad()
        output = model(img)

        loss = loss_fun(pred, tar)
        loss.backward()

        _, pred = torch.max(output, 1)

        correct += (pred == tar).sum().item()
        total += tar.shape[0]
    
    acc = (correct/total) *100
    print(acc)


In [ ]:
total_test = 0
correct_test = 0
with torch.no_grad():
 for img, label in testLoaderr:
    output_test = model(img)
    _, pred = torch.max(output_test, 1)
    correct_test += (pred == label).sum().item()
    total_test += label.shape[0]


acc = (correct_test/total_test) * 100
print(acc)

